In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql import functions as F
import matplotlib.pyplot as plt 
import seaborn as sns 

In [2]:
spark = SparkSession.builder\
        .appName('traitement_NLP').getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/24 19:02:15 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
spark.version

'3.5.7'

# load des données 

In [4]:
### ---------------------- nos liens --------------------------------------------####
url = "jdbc:postgresql://postgres_warehouse:5432/Warehouse_DB"
user = "admin"
password = "admin_pwd"
driver = "org.postgresql.Driver"

In [5]:
data = spark.read.format("jdbc")\
            .option("url", url).option("dbtable", "dataset_ml_propore")\
            .option("user", user).option("password", password).option("driver", driver).load()
       

In [6]:
data.show(5)

+----------------+--------------------+--------------------+--------------+--------------------+--------------------+--------+--------------------+-------------+-------------------+
|      entreprise|               poste|                lien|    competence|           formation|        niveau_etude|contract|          experience|       region|date_de_publication|
+----------------+--------------------+--------------------+--------------+--------------------+--------------------+--------+--------------------+-------------+-------------------+
|INTRA INTERIM SN|Ouvrier d’Usine -...|https://www.emplo...|Accompagnement|Niveau d’anglais ...|Qualification ava...|     CDI|            Etudiant|International|         20.06.2025|
|INTRA INTERIM SN|Ouvrier d’Usine -...|https://www.emplo...|Accompagnement|Niveau d’anglais ...|Qualification ava...|     CDI|jeune diplômé et ...|International|         20.06.2025|
|INTRA INTERIM SN|Ouvrier d’Usine -...|https://www.emplo...|Accompagnement|Niveau d’anglai

In [11]:
data.columns

['entreprise',
 'poste',
 'lien',
 'competence',
 'formation',
 'niveau_etude',
 'contract',
 'experience',
 'region',
 'date_de_publication']

In [12]:
data.printSchema()

root
 |-- entreprise: string (nullable = true)
 |-- poste: string (nullable = true)
 |-- lien: string (nullable = true)
 |-- competence: string (nullable = true)
 |-- formation: string (nullable = true)
 |-- niveau_etude: string (nullable = true)
 |-- contract: string (nullable = true)
 |-- experience: string (nullable = true)
 |-- region: string (nullable = true)
 |-- date_de_publication: string (nullable = true)



#### Notre dataset est constitué de données qualitatives, le feature engenieering s'impose pour faire nos etude 

### Affichage des differents colonnes

In [13]:
data.select(data.entreprise).show(2, truncate=False) 

+----------------+
|entreprise      |
+----------------+
|INTRA INTERIM SN|
|INTRA INTERIM SN|
+----------------+
only showing top 2 rows



In [15]:
data.select(data.poste).distinct().show(2, truncate=False) 

+-----------------------------------------------------------------------------------+
|poste                                                                              |
+-----------------------------------------------------------------------------------+
|Agent de Confection et d´Assemblage - Dakar / Almadies                             |
|Expert en Matériaux Biosourcés – Formation d´Ouvriers sur Chantier  - Thionk Essyl |
+-----------------------------------------------------------------------------------+
only showing top 2 rows



In [16]:
data.select(data.competence).distinct().show(2, truncate=False) 

+----------+
|competence|
+----------+
|LARAVEL   |
|C#        |
+----------+
only showing top 2 rows



In [17]:
data.select(data.formation).distinct().show(2, truncate=False) 

+---------------------------------------------------------------------------+
|formation                                                                  |
+---------------------------------------------------------------------------+
|Formation : Niveau Bac + 2 ou plus                                         |
|Français parfaitement maîtrisé à l’oral comme à l’écrit, avec accent neutre|
+---------------------------------------------------------------------------+
only showing top 2 rows



In [18]:
data.select(data.niveau_etude).distinct().show(2, truncate=False) 

+------------+
|niveau_etude|
+------------+
|Bac         |
|Bac+2       |
+------------+
only showing top 2 rows



In [19]:
data.select(data.contract).distinct().show(2, truncate=False) 

+-------------+
|contract     |
+-------------+
|Temps partiel|
|CDI          |
+-------------+
only showing top 2 rows



In [20]:
data.select(data.experience).distinct().show(2, truncate=False) 

+-----------------------------+
|experience                   |
+-----------------------------+
|2-5 ans                      |
|jeune diplômé Sans expérience|
+-----------------------------+
only showing top 2 rows



In [21]:
data.select(data.region).distinct().show(2, truncate=False) 

+---------------------+
|region               |
+---------------------+
|International        |
|Dakar & International|
+---------------------+
only showing top 2 rows



## verification du netoyages des données 

In [23]:
data.describe()

DataFrame[summary: string, entreprise: string, poste: string, lien: string, competence: string, formation: string, niveau_etude: string, contract: string, experience: string, region: string, date_de_publication: string]

In [30]:
data.filter(col("poste").isNull()).count()

0

In [39]:
colonnes = ["poste","lien","competence","formation","niveau_etude","contract","experience","region","date_de_publication"]
for el in colonnes:
    print(el,data.filter(col(el).isNull()).count())

poste 0
lien 0
competence 0
formation 0
niveau_etude 0
contract 0
experience 0
region 0
date_de_publication 0


# Features engineering